# Lesson 4a — Counts model baseline (no neural net at all)

Phase 2 of the Karpathy track: build PRAGMA incrementally. Same data and task throughout L4a-L4d, with one piece added each lesson.

This is the baseline.

Runnable version of `04a_*.py`.


## The dataset (used by L4a–L4d)

In [ ]:
import math, random
from collections import Counter, defaultdict
random.seed(0)

KEYS   = ["pet", "action", "place"]
VALUES = ["dog", "cat", "fish", "eat", "sleep", "play", "garden", "couch", "bowl"]
RULES = {
    "dog":  {"action": ["eat", "play"],   "place": ["garden", "bowl"]},
    "cat":  {"action": ["sleep", "play"], "place": ["couch", "bowl"]},
    "fish": {"action": ["eat", "sleep"],  "place": ["bowl"]},
}

def random_event():
    pet = random.choice(list(RULES))
    act = random.choice(RULES[pet]["action"])
    plc = random.choice(RULES[pet]["place"])
    return {"pet": pet, "action": act, "place": plc}

def make_dataset(n):
    return [random_event() for _ in range(n)]

train = make_dataset(5000)
test  = make_dataset(1000)
print(f"Train: {len(train)}, Test: {len(test)}")
print(f"Example: {train[0]}")

## The counts model — pure bookkeeping, no neural net

For each `(visible_key, visible_value, target_key)` triple, count how often each target value appeared in training. At inference time, average the per-feature distributions.

In [ ]:
counts = defaultdict(Counter)
for ev in train:
    for vis_k in KEYS:
        for tgt_k in KEYS:
            if vis_k == tgt_k:
                continue
            counts[(vis_k, ev[vis_k], tgt_k)][ev[tgt_k]] += 1

def probs_for(vis_k, vis_v, tgt_k):
    c = counts[(vis_k, vis_v, tgt_k)]
    total = sum(c.values())
    if total == 0:
        return {v: 1.0 / len(VALUES) for v in VALUES}
    return {v: cnt / total for v, cnt in c.items()}

def counts_predict(visible, target_key):
    distribs = [probs_for(k, v, target_key) for k, v in visible.items()]
    avg = defaultdict(float)
    for d in distribs:
        for k, v in d.items():
            avg[k] += v / len(distribs)
    return dict(avg)

# Inspect
print(f"P(place | pet=dog):  {probs_for('pet', 'dog', 'place')}")
print(f"P(place | pet=cat):  {probs_for('pet', 'cat', 'place')}")

## Evaluate

Hide each field in turn, score accuracy + cross-entropy.

In [ ]:
def evaluate(model_predict, dataset):
    correct = 0
    total = 0
    losses = []
    for ev in dataset:
        for tgt_k in KEYS:
            visible = {k: v for k, v in ev.items() if k != tgt_k}
            probs = model_predict(visible, tgt_k)
            true_v = ev[tgt_k]
            pred_v = max(probs.items(), key=lambda kv: kv[1])[0]
            correct += int(pred_v == true_v)
            total += 1
            p_true = probs.get(true_v, 1e-9)
            losses.append(-math.log(max(p_true, 1e-9)))
    return correct / total, sum(losses) / len(losses)

acc, loss = evaluate(counts_predict, test)
print(f"  Test accuracy:        {acc * 100:.1f}%")
print(f"  Test cross-entropy:   {loss:.3f}")
print()
print("This is our BASELINE. Every subsequent lesson should beat it.")

## Next

[**L4b**](lesson_04b_embedding_linear.ipynb) — replace counts with learned embeddings + a linear head. Same data, same task, more powerful model.